# directory-pipeline — Surya OCR + Alignment Review

Runs Surya OCR and/or the interactive alignment review tool on Google Colab.

**Before you start:**
- Enable a GPU runtime: Runtime → Change runtime type → T4 GPU (required for Surya)
- If using ngrok as a fallback tunnel, store your token as a Colab secret named `NGROK_TOKEN` (Secrets panel in the left sidebar)

Run the **Setup** cells first, then jump to whichever section you need.

---
## Setup

Run these cells once per session.

In [1]:
# --- Static config — edit once ---
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"
MODEL = "gemini-2.0-flash"
PORT  = 5000

In [2]:
import ipywidgets as widgets
from IPython.display import display

slug_widget = widgets.Text(
    value="",
    placeholder="e.g. brewers_guide_for_the_united_states_cana_bd047d10",
    description="Volume slug:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

def _on_change(change):
    global VOLUME
    VOLUME = f"output/{change['new']}"
    print(f"VOLUME set to: {VOLUME}")

slug_widget.observe(_on_change, names="value")
display(slug_widget)

Text(value='', description='Volume slug:', layout=Layout(width='600px'), placeholder='e.g. brewers_guide_for_t…

VOLUME set to: output/go_guide_to_pleasant_motoring_e6743f00


In [4]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.chdir(PIPELINE_DIR)
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/Othercomputers/My_Mac/directory-pipeline


In [5]:
# Install dependencies (~2 min on first run — surya-ocr pulls in torch/transformers).
# Colab already has numpy; torch is reinstalled by surya-ocr at the right CUDA version.
%pip install -q google-genai pillow requests python-dotenv flask geopy pyngrok

In [8]:
!pip install "surya-ocr==0.17.1" "transformers>=4.56.1,<5.0"


  Using cached surya_ocr-0.17.1-py3-none-any.whl.metadata (34 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
Using cached surya_ocr-0.17.1-py3-none-any.whl (189 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)


---
## Run Surya OCR

Runs Surya bounding-box detection on images in an `output/` volume directory.
Produces `*_surya.json` files used by the alignment step.

Adjust `--batch-size` to fit your GPU memory (T4: 8–16 for regular images, higher for microfilm).

In [9]:
# Regular photographic / scan materials
# Set VOLUME_SLUG in the config cell above, then run this.
#!time python main.py --batch-size 16 --surya-ocr {VOLUME}

In [10]:
# Microfilm / high-contrast materials — larger batches fit in memory
# Uncomment to use instead of the regular cell above.
# !time python main.py --batch-size 64 --surya-ocr {VOLUME}

---
## Run alignment

Aligns Surya bounding boxes with Gemini OCR text. Requires `*_surya.json` and
`*_gemini.json` files to already exist in the volume directory.

In [11]:
#!time python main.py --align-ocr --model {MODEL} {VOLUME}

---
## Review alignment

Starts the interactive Flask review tool and exposes it to your browser.

**Preferred**: Colab's built-in proxy (no account needed) — run the _Start server_ and _Open via Colab proxy_ cells.

**Fallback**: If the proxy URL doesn't work, use the ngrok cells instead.

In [19]:
import os
os.path(VOLUME)

TypeError: 'module' object is not callable

In [12]:
import subprocess

subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)


subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server starting on {VOLUME} — run the next cell to watch the log.")

Server starting on output/go_guide_to_pleasant_motoring_e6743f00 — run the next cell to watch the log.


In [15]:
# Watch the log. Stop this cell (square button) once you see "Models ready."
!tail -f /tmp/review.log

Error: directory not found: output/go_guide_to_pleasant_motoring_e6743f00
^C


In [ ]:
# Option A: Colab built-in proxy (no account needed — try this first)
from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Review tool URL:", url)

In [ ]:
# # Option B: ngrok tunnel (fallback if the proxy URL doesn't work)
# # Requires an NGROK_TOKEN secret set in the Colab Secrets panel (left sidebar).
# from pyngrok import ngrok
# from google.colab import userdata

# for t in ngrok.get_tunnels():
#     ngrok.disconnect(t.public_url)

# ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
# public_url = ngrok.connect(PORT)
# print("Review tool URL:", public_url)

---
## Troubleshooting

**Server not responding** — check the log:
```python
!cat /tmp/review.log
```

**Port already in use** — kill it and restart the server cell:
```python
!fuser -k 5000/tcp
```

**`ERR_NGROK_324` — too many tunnels** — kill old ones, then re-run the ngrok cell:
```python
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
```

**Clicking Done in the UI shuts down the server.** To restart, re-run the _Start server_ and _Open via proxy/ngrok_ cells.

In [ ]:
import subprocess
from pathlib import Path

base = Path(PIPELINE_DIR) / "output/green_books_and_related"

for collection in sorted(base.iterdir()):
    if not collection.is_dir() or collection.name.startswith("_"):
        continue
    print(f"\n{'='*60}\n{collection.name}\n{'='*60}")
    result = subprocess.run(
        ["python", "pipeline/run_surya_ocr.py", str(collection), "--batch-size", "128"],
        cwd=PIPELINE_DIR,
    )



go_guide_to_pleasant_motoring_e6743f00

hackley_harrison_s_hotel_and_apartment_g_4f7822b0

n_h_a_directory_and_guide_to_travelers_b33397d0

smith_s_tourist_guide_of_necessary_infor_e20bf5b0

the_green_book_9ea5d5b0

the_travelers_guide_e088efa0

travelguide_634f3af0



go_guide_to_pleasant_motoring_e6743f00

hackley_harrison_s_hotel_and_apartment_g_4f7822b0

n_h_a_directory_and_guide_to_travelers_b33397d0

smith_s_tourist_guide_of_necessary_infor_e20bf5b0

the_green_book_9ea5d5b0

the_travelers_guide_e088efa0

travelguide_634f3af0
Server starting on output/go_guide_to_pleasant_motoring_e6743f00 — run the next cell to watch the log.


In [ ]:
import subprocess
from pathlib import Path


result = subprocess.run(
    ["python", "pipeline/run_surya_ocr.py", str(collection), "--batch-size", "4"],
    cwd=PIPELINE_DIR,
    capture_output=True,
    text=True,
)
Path("/tmp/surya_err.txt").write_text(result.stderr)
print("Return code:", result.returncode)
print("STDOUT:", result.stdout)


NameError: name 'collection' is not defined

In [21]:
!tail -50 /tmp/surya_err.txt

tail: cannot open '/tmp/surya_err.txt' for reading: No such file or directory


All volumes

In [28]:
import subprocess

subprocess.run([
    "rsync", "-av", "--progress",
    "--include=*/",
    "--include=*_aligned.json",
    "--include=*.jpg",
    "--exclude=*_viz.jpg",
    "--exclude=*",
    "/content/drive/Othercomputers/My_Mac/directory-pipeline/output/green_books_and_related/",
    "/content/review_work/"
])


KeyboardInterrupt: 

In [ ]:
!du -sh /content/review_work/


In [24]:
import subprocess
from google.colab.output import eval_js

#OUTPUT_DIR = f"{PIPELINE_DIR}/output/green_books_and_related"
OUTPUT_DIR = "/content/review_work"

subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)

subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     OUTPUT_DIR,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"Server starting — run the next cell to watch the log.")
print(f"URL: {eval_js(f'google.colab.kernel.proxyPort({PORT})')}")


Server starting — run the next cell to watch the log.
URL: https://5000-gpu-l4-s-kkb-ass1a1-3fv4d146y41o4-a.asia-southeast1-1.prod.colab.dev


In [26]:
!cat /tmp/review.log | tail -50

  images: /content/drive/Othercomputers/My_Mac/directory-pipeline/output/green_books_and_related
  model:  gemini-2.0-flash
Loading Surya models… (this takes ~30 s)
2026-03-17 01:43:01.099465: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 01:43:01.118344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773711781.141904    8597 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773711781.149683    8597 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has al

Ngrok version
